# 05. Продолжение обучения детекции изображений

Этот ноутбук продолжает прерванный запуск, не изменяя его папку. По умолчанию указан запуск, в котором оборвалось соединение. Подключите новый Colab runtime с T4 и выполните ячейки сверху вниз. Если продолжаете уже этот новый запуск после ещё одного обрыва, укажите его ID в настройках ниже.

Готовые модели используются повторно, Faster R-CNN продолжает с последней сохранённой эпохи; остальные модели обучаются по прежним настройкам. Сохраняются две последние контрольные точки Faster R-CNN. Новая папка результатов и ID появятся в выводе.

У старых чекпоинтов Faster R-CNN не сохранено состояние генераторов случайных чисел: веса и оптимизатор восстанавливаются, но порядок примеров после обрыва может отличаться от непрерывного обучения. Проверьте, что старый runtime остановлен, прежде чем читать его файлы.


In [1]:
import importlib.util
from importlib import metadata as importlib_metadata
import subprocess
import sys
needed = []
for module, package in [('ultralytics', 'ultralytics==8.3.218'), ('pycocotools', 'pycocotools>=2.0.7,<3'),
                        ('sklearn', 'scikit-learn>=1.4,<2'), ('yaml', 'PyYAML>=6,<7'),
                        ('PIL', 'Pillow>=10,<14'), ('pandas', 'pandas>=2.2,<3'),
                        ('matplotlib', 'matplotlib>=3.8,<4')]:
    if importlib.util.find_spec(module) is None:
        needed.append(package)
    elif module == 'ultralytics' and importlib_metadata.version('ultralytics') != '8.3.218':
        needed.append(package)
if needed:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *needed])
    importlib.invalidate_caches()
print('Зависимости доступны.')


Зависимости доступны.


## 1. Настройки и пути

Для каждой архитектуры используются одинаковые настройки и одинаковые начальные предобученные веса во всех четырёх вариантах. Число эпох выбрано как стартовое; до основного эксперимента его можно изменить для всех вариантов модели сразу. Модель выбирается по **последней эпохе**, а не по test.


In [2]:
from pathlib import Path
import os
import json
import uuid
import tempfile
from datetime import datetime, timezone
SEED = 42
YOLO_EPOCHS = 10
FRCNN_EPOCHS = 10
IMG_SIZE = 640
YOLO_BATCH = 8
FRCNN_BATCH = 2
CONFIDENCE = 0.25
MATCH_IOU = 0.5

local_root = os.environ.get('COURSEWORK_DATA_ROOT')
if local_root:
    ROOT = Path(local_root).expanduser().resolve()
else:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError('Выберите Colab runtime в VS Code.') from exc
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/data-enrichment-coursework')
# Измените ROOT, если папка проекта находится в другом месте Drive.
PATHS = {
    'raw_images': ROOT / 'raw/images/images.tar.gz',
    'manifest': ROOT / 'processed/images/pet_head_clean_manifest.csv',
    'split': ROOT / 'splits/images/oxford_pet_split_seed42.csv',
    'split_meta': ROOT / 'splits/images/oxford_pet_split_seed42.json',
    'manual_annotations': ROOT / 'manual/images/annotations_final.json',
    'manual_images': ROOT / 'manual/images/main',
    'augmented_zip': ROOT / 'augmented/images/augmented_images.zip',
}
missing = [str(p) for p in PATHS.values() if not p.exists()]
if missing:
    raise FileNotFoundError('Не найдены файлы / папки:\n' + '\n'.join(missing) +
                            '\nПроверьте ROOT и ярлык общей папки в «Мой диск».')
print('Папка проекта:', ROOT)
for name, path in PATHS.items(): print(name, '→', path)

# ID запуска, который нужно продолжить. Для следующего возобновления укажите ID нового запуска.
RESUME_SOURCE_RUN_ID = '20261118T103517Z_8430340a'
SOURCE_RUN_DIR = ROOT / 'results/images/training' / RESUME_SOURCE_RUN_ID
if not SOURCE_RUN_DIR.is_dir():
    raise FileNotFoundError(f'Не найдена папка прерванного запуска: {SOURCE_RUN_DIR}. Проверьте ID и Google Drive.')
print('Источник восстановления:', SOURCE_RUN_DIR)
# Сохраняем цепочку запусков, чтобы найти все готовые модели после нескольких обрывов.
SOURCE_RUNS = []
cur = SOURCE_RUN_DIR
while True:
    if cur in SOURCE_RUNS: raise RuntimeError('Цикл в цепочке восстановлений.')
    SOURCE_RUNS.append(cur)
    link = cur / 'recovery_parent.json'
    if not link.exists(): break
    parent_id = json.loads(link.read_text(encoding='utf-8'))['source_run_id']
    if Path(parent_id).name != parent_id: raise ValueError('Некорректный ID предыдущего запуска.')
    cur = ROOT / 'results/images/training' / parent_id
    if not cur.is_dir(): raise FileNotFoundError(f'Нет предыдущей папки запуска: {cur}')
print('Цепочка папок:', *SOURCE_RUNS, sep='\n  ')


Mounted at /content/drive
Папка проекта: /content/drive/MyDrive/data-enrichment-coursework
raw_images → /content/drive/MyDrive/data-enrichment-coursework/raw/images/images.tar.gz
manifest → /content/drive/MyDrive/data-enrichment-coursework/processed/images/pet_head_clean_manifest.csv
split → /content/drive/MyDrive/data-enrichment-coursework/splits/images/oxford_pet_split_seed42.csv
split_meta → /content/drive/MyDrive/data-enrichment-coursework/splits/images/oxford_pet_split_seed42.json
manual_annotations → /content/drive/MyDrive/data-enrichment-coursework/manual/images/annotations_final.json
manual_images → /content/drive/MyDrive/data-enrichment-coursework/manual/images/main
augmented_zip → /content/drive/MyDrive/data-enrichment-coursework/augmented/images/augmented_images.zip
Источник восстановления: /content/drive/MyDrive/data-enrichment-coursework/results/images/training/20261118T074256Z_475488e0
Цепочка папок:
  /content/drive/MyDrive/data-enrichment-coursework/results/images/train

## 2. Проверка исходных данных и выделение test

Из 734 изображений исходной validation-части выделяются 367 в validation и 367 в test, стратифицированно по породе. Фиксируются ID и параметры разбиения. При обучении используются только исходный train и его разрешённые дополнения; test виден только на финальной оценке.


In [3]:
import csv
import hashlib
import io
import json
import random
import shutil
import tarfile
import time
import zipfile
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
import yaml
try:
    from IPython.display import display
except ImportError:
    def display(value): print(value)

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()

def fail_if(condition, message):
    if condition: raise ValueError(message)

manifest = pd.read_csv(PATHS['manifest'])
split = pd.read_csv(PATHS['split'])
meta = json.loads(PATHS['split_meta'].read_text(encoding='utf-8'))
fail_if(sha256_file(PATHS['manifest']) != meta['input_sha256'], 'Хеш манифеста Oxford Pet не совпадает с разбиением.')
fail_if(sha256_file(PATHS['split']) != meta['split_sha256'], 'Хеш файла разбиения Oxford Pet не совпадает.')
fail_if(len(manifest) != 3670 or manifest['id'].duplicated().any(), 'Манифест должен содержать 3670 уникальных изображений.')
fail_if(len(split) != len(manifest) or split['id'].duplicated().any() or set(split['id']) != set(manifest['id']),
        'ID разбиения и манифеста не совпадают.')
base = manifest.merge(split, on='id', validate='one_to_one', suffixes=('', '_split'))
for column in ('breed', 'species', 'image_sha256'):
    fail_if(not base[column].eq(base[column + '_split']).all(), f'Разбиение не совпадает по {column}.')
fail_if(set(base['split']) != {'train', 'validation'}, 'Ожидается исходное разбиение train/validation.')
base = base.sort_values('id').reset_index(drop=True)
holdout = base.loc[base['split'].eq('validation')].copy()
train_base = base.loc[base['split'].eq('train')].copy()
val_ids, test_ids = train_test_split(holdout['id'].to_numpy(), test_size=0.5, random_state=SEED,
                                     stratify=holdout['breed'].to_numpy())
val_ids, test_ids = set(val_ids), set(test_ids)
train_ids = set(train_base['id'])
fail_if(bool((train_ids & val_ids) | (train_ids & test_ids) | (val_ids & test_ids)), 'Пересечение ID между частями!')
fail_if(len(train_ids) != 2936 or len(val_ids) != 367 or len(test_ids) != 367, 'Неожиданный размер разбиения.')
final_split = base[['id', 'breed', 'species', 'image_sha256']].copy()
final_split['split'] = final_split['id'].map(lambda x: 'train' if x in train_ids else ('validation' if x in val_ids else 'test'))
# Возвращаем именно сохранённое разбиение старого запуска, даже при другой версии sklearn.
old_split_path = SOURCE_RUN_DIR / 'evaluation_split.csv'
if not old_split_path.is_file(): raise FileNotFoundError(f'Нет сохранённого разбиения: {old_split_path}')
old_split = pd.read_csv(old_split_path)
columns = ['id', 'breed', 'species', 'image_sha256']
fail_if(old_split['id'].duplicated().any() or set(old_split['id']) != set(final_split['id']), 'ID сохранённого разбиения не совпадают.')
check = old_split.merge(final_split[columns], on='id', suffixes=('_old', '_new'), validate='one_to_one')
for col in columns[1:]:
    fail_if(not check[col+'_old'].eq(check[col+'_new']).all(), f'Сохранённое разбиение не совпадает по {col}.')
fail_if(set(old_split['split']) != {'train','validation','test'}, 'Некорректные части сохранённого разбиения.')
fail_if(set(old_split.loc[old_split['split'].eq('train'),'id']) != train_ids, 'Изменилась исходная обучающая часть.')
val_ids = set(old_split.loc[old_split['split'].eq('validation'),'id'])
test_ids = set(old_split.loc[old_split['split'].eq('test'),'id'])
fail_if(len(val_ids)!=367 or len(test_ids)!=367 or (val_ids | test_ids)!=set(holdout['id']), 'Некорректная сохранённая оценочная часть.')
final_split = old_split[columns+['split']].copy()
fail_if(final_split.groupby('image_sha256')['split'].nunique().max() > 1, 'Одинаковые исходные файлы попали в разные части!')
manual = json.loads(PATHS['manual_annotations'].read_text(encoding='utf-8'))
fail_if(len(manual) != 100 or len({r['id'] for r in manual}) != 100, 'Нужно 100 уникальных ручных рамок.')
for r in manual:
    fail_if(r['label'] != 'pet_head' or r['image_filename'] != r['id'] + '.jpg', f"Неверный класс или имя ручного изображения: {r['id']}")
with zipfile.ZipFile(PATHS['augmented_zip']) as z:
    ox_flips = json.loads(z.read('oxford_pet_horizontal_flip/annotations.json'))
    manual_flips = json.loads(z.read('manual_horizontal_flip/annotations.json'))
ox_flips = {r['parent_id']: r for r in ox_flips}
manual_flips = {r['parent_id']: r for r in manual_flips}
fail_if(len(ox_flips) != len(base) or set(ox_flips) != set(base['id']), 'Набор отражений Oxford неполон.')
fail_if(len(manual_flips) != len(manual) or set(manual_flips) != {r['id'] for r in manual}, 'Набор ручных отражений неполон.')
fail_if(set(manual_flips) & set(base['id']), 'ID ручных изображений пересекаются с Oxford.')
for row in base.itertuples(index=False):
    f = ox_flips[row.id]
    fail_if(f['label'] != 'pet_head' or f['split'] != row.split or f['parent_sha256'] != row.image_sha256, f'Отражение Oxford не соответствует оригиналу: {row.id}')
    fail_if(bool(f['eligible_for_training']) != (row.split == 'train') or f['eligible_for_evaluation'] is not False,
            f'Некорректное назначение отражения: {row.id}')
print('Oxford train / validation / test:', len(train_ids), len(val_ids), len(test_ids))
print('Ручные изображения для обучения:', len(manual))


Oxford train / validation / test: 2936 367 367
Ручные изображения для обучения: 100


## 3. Подготовка изображений и рамок

На временный диск копируются исходный архив изображений и архив отражений. Оригиналы test и validation не преобразуются. Рамки приводятся к пиксельному формату `xyxy`: XML Oxford трактуется как Pascal VOC с единицы и включёнными границами; ручные координаты в процентах переводятся в пиксели. Хеши изображений и соответствие отражённых рамок проверяются до обучения.


In [4]:
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
RUN_DIR = ROOT / 'results/images/training' / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
work_parent = Path(os.environ.get('COURSEWORK_WORK_DIR', '/content' if Path('/content').exists() else tempfile.gettempdir()))
work_parent.mkdir(parents=True, exist_ok=True)
WORK = Path(tempfile.mkdtemp(prefix='coursework_pet_', dir=work_parent))
image_dir = WORK / 'images'
label_dir = WORK / 'labels'
image_dir.mkdir(); label_dir.mkdir()
final_split.to_csv(RUN_DIR / 'evaluation_split.csv', index=False)
(RUN_DIR/'recovery_parent.json').write_text(json.dumps({'source_run_id':RESUME_SOURCE_RUN_ID}),encoding='utf-8')
print('ID нового запуска для повторного восстановления:', RUN_ID)
print('Рабочие файлы:', WORK)
print('Результаты:', RUN_DIR)
print('Копирование архивов на временный диск...')
raw_local = WORK / 'images.tar.gz'
aug_local = WORK / 'augmented_images.zip'
shutil.copyfile(PATHS['raw_images'], raw_local)
shutil.copyfile(PATHS['augmented_zip'], aug_local)

records = {}
def register(record):
    key = record['id']
    fail_if(key in records, f'Повтор ID: {key}')
    x1,y1,x2,y2 = record['box']; w,h = record['width'],record['height']
    fail_if(not (0 <= x1 < x2 <= w and 0 <= y1 < y2 <= h), f'Рамка вне изображения: {key}')
    fail_if(not record['path'].is_file(), f'Изображение не найдено: {key}')
    records[key] = record
    # YOLO: класс 0, затем центр и размер в долях ширины и высоты.
    label = f'0 {(x1+x2)/(2*w):.10f} {(y1+y2)/(2*h):.10f} {(x2-x1)/w:.10f} {(y2-y1)/h:.10f}\n'
    (label_dir / (key + '.txt')).write_text(label, encoding='utf-8')

base_rows = {r['id']: r for r in base.to_dict('records')}
part_by_id = dict(zip(final_split['id'], final_split['split']))
found = set()
with tarfile.open(raw_local, mode='r|gz') as tar:
    for member in tar:
        if not member.isfile() or Path(member.name).suffix.lower() != '.jpg': continue
        key = Path(member.name).stem
        if key not in base_rows: continue
        fail_if(key in found, f'Повтор в архиве: {key}')
        raw = tar.extractfile(member).read(); row = base_rows[key]
        fail_if(hashlib.sha256(raw).hexdigest() != row['image_sha256'], f'Хеш исходного JPG не совпадает: {key}')
        with Image.open(io.BytesIO(raw)) as image:
            fail_if(image.size != (int(row['width']), int(row['height'])), f'Размер не совпадает: {key}')
            fail_if(image.getexif().get(274, 1) not in (0, 1), f'Неожиданная EXIF-ориентация: {key}')
        dst = image_dir / ('ox_' + key + '.jpg'); dst.write_bytes(raw)
        x1,y1,x2,y2 = [int(float(row[k])) for k in ('xmin_xml', 'ymin_xml', 'xmax_xml', 'ymax_xml')]
        register({'id':'ox_' + key, 'parent_id':key, 'kind':'oxford_original', 'split':part_by_id[key],
                  'path':dst, 'width':int(row['width']), 'height':int(row['height']),
                  'box':[x1-1, y1-1, x2, y2], 'sha256':row['image_sha256']})
        found.add(key)
fail_if(found != set(base_rows), f'В архиве не найдены изображения: {len(set(base_rows)-found)}')
print('Исходные Oxford:', len(found))

manual_by_id = {r['id']:r for r in manual}
for key, row in sorted(manual_by_id.items()):
    source = PATHS['manual_images'] / key.split('_')[0] / row['image_filename']
    fail_if(not source.is_file(), f'Нет ручной фотографии: {source}')
    f = manual_flips[key]
    fail_if(sha256_file(source) != f['parent_sha256'], f'Хеш ручной фотографии не совпадает: {key}')
    w,h = int(row['image_width']), int(row['image_height'])
    with Image.open(source) as image:
        fail_if(image.size != (w,h) or image.getexif().get(274,1) not in (0,1), f'Размер / ориентация не совпадает: {key}')
    dst = image_dir / ('manual_' + key + '.jpg'); shutil.copyfile(source, dst)
    v = row['bbox_percent_xyxy']; box = [v['x_min']*w/100, v['y_min']*h/100, v['x_max']*w/100, v['y_max']*h/100]
    register({'id':'manual_' + key,'parent_id':key,'kind':'manual_original','split':'train','path':dst,
              'width':w,'height':h,'box':box,'sha256':f['parent_sha256']})
print('Ручные оригиналы:', len(manual))

with zipfile.ZipFile(aug_local) as z:
    for key, flip in sorted(ox_flips.items()):
        if key not in train_ids: continue
        row=base_rows[key]; w,h=int(row['width']),int(row['height'])
        x1,y1,x2,y2=[int(float(row[k])) for k in ('xmin_xml','ymin_xml','xmax_xml','ymax_xml')]
        box=[w-x2,y1-1,w-x1+1,y2]
        b=flip['bbox_xyxy_zero_based_half_open']
        fail_if(box != [b['x_min'],b['y_min'],b['x_max'],b['y_max']], f'Некорректная отражённая рамка: {key}')
        fail_if((flip['image_width'],flip['image_height']) != (w,h), f'Размер отражения не совпадает: {key}')
        raw=z.read('oxford_pet_horizontal_flip/' + flip['image_filename'])
        fail_if(hashlib.sha256(raw).hexdigest() != flip['image_sha256'], f'Хеш отражения не совпадает: {key}')
        dst=image_dir / ('ox_' + key + '_hflip.png'); dst.write_bytes(raw)
        register({'id':'ox_' + key + '_hflip','parent_id':key,'kind':'oxford_flip','split':'train',
                  'path':dst,'width':w,'height':h,'box':box,'sha256':flip['image_sha256']})
    for key, flip in sorted(manual_flips.items()):
        row=manual_by_id[key]; w,h=int(row['image_width']),int(row['image_height'])
        fail_if(flip['split'] != 'train' or flip['parent_split'] != 'train' or not flip['eligible_for_training'] or flip['eligible_for_evaluation'], f'Назначение ручного отражения: {key}')
        original=row['bbox_percent_xyxy']; b=flip['bbox_percent_xyxy']
        expected=[100-original['x_max'],original['y_min'],100-original['x_min'],original['y_max']]
        actual=[b['x_min'],b['y_min'],b['x_max'],b['y_max']]
        fail_if(not all(abs(a-c)<1e-8 for a,c in zip(actual,expected)), f'Некорректная ручная рамка: {key}')
        fail_if((flip['image_width'],flip['image_height']) != (w,h), f'Размер ручного отражения: {key}')
        raw=z.read('manual_horizontal_flip/' + flip['image_filename'])
        fail_if(hashlib.sha256(raw).hexdigest() != flip['image_sha256'], f'Хеш ручного отражения: {key}')
        dst=image_dir / ('manual_' + key + '_hflip.png'); dst.write_bytes(raw)
        box=[b['x_min']*w/100,b['y_min']*h/100,b['x_max']*w/100,b['y_max']*h/100]
        register({'id':'manual_' + key + '_hflip','parent_id':key,'kind':'manual_flip','split':'train',
                  'path':dst,'width':w,'height':h,'box':box,'sha256':flip['image_sha256']})
print('Изображений с рамками подготовлено:', len(records))


ID нового запуска для повторного восстановления: 20261118T103517Z_8430340a
Рабочие файлы: /content/coursework_pet_g1dwd0nv
Результаты: /content/drive/MyDrive/data-enrichment-coursework/results/images/training/20261118T103517Z_8430340a
Копирование архивов на временный диск...
Исходные Oxford: 3670
Ручные оригиналы: 100
Изображений с рамками подготовлено: 6806


## 4. Четыре обучающих варианта

Validation и test во всех вариантах одинаковы и содержат только оригиналы Oxford Pet. Отражённые копии закреплены за исходными изображениями и разрешены только для train. Варианты различаются объёмом данных: сравнение отражает совместный эффект состава и количества примеров.


In [5]:
variants = {}
for name, kinds in {
    'baseline': {'oxford_original'},
    'manual': {'oxford_original','manual_original'},
    'flipped': {'oxford_original','oxford_flip'},
    'manual_flipped': {'oxford_original','manual_original','oxford_flip','manual_flip'},
}.items():
    variants[name] = [r for r in records.values() if r['split']=='train' and r['kind'] in kinds]
variant_labels = {'baseline':'Исходные', 'manual':'Исходные + ручные',
                  'flipped':'Исходные + отражения', 'manual_flipped':'Исходные + ручные + отражения'}
val_records = [r for r in records.values() if r['split']=='validation']
test_records = [r for r in records.values() if r['split']=='test']
fail_if(len(val_records)!=367 or len(test_records)!=367, 'Некорректная оценочная выборка.')
fail_if(any(r['kind']!='oxford_original' for r in val_records+test_records), 'Оценка должна использовать только оригиналы.')
heldout_ids = {r['parent_id'] for r in val_records+test_records}
for name, rows in variants.items():
    fail_if(bool({r['parent_id'] for r in rows} & heldout_ids), f'Утечка пар в варианте {name}!')
expected = {'baseline':2936,'manual':3036,'flipped':5872,'manual_flipped':6072}
fail_if({k:len(v) for k,v in variants.items()} != expected, 'Неожиданный размер тренировочного варианта.')

def write_list(path, rows):
    path.write_text(''.join(str(r['path'].resolve()) + '\n' for r in rows), encoding='utf-8')
write_list(WORK/'validation.txt', val_records); write_list(WORK/'test.txt', test_records)
for name, rows in variants.items():
    write_list(WORK/(name+'_train.txt'), rows)
    config={'path':str(WORK),'train':str(WORK/(name+'_train.txt')),
            'val':str(WORK/'validation.txt'),'test':str(WORK/'test.txt'),'names':{0:'pet_head'}}
    (WORK/(name+'.yaml')).write_text(yaml.safe_dump(config, allow_unicode=True),encoding='utf-8')
    pd.DataFrame([{'id':r['id'],'parent_id':r['parent_id'],'kind':r['kind'],'sha256':r['sha256']} for r in rows]).to_csv(RUN_DIR/(name+'_training_ids.csv'),index=False)
display(pd.DataFrame([{'вариант':variant_labels[k],'train':len(v),'validation':len(val_records),'test':len(test_records)} for k,v in variants.items()]))

def verify_previous_run():
    # Старые списки фиксируют ID и хеши изображений. Хеши рамок старый код не сохранял.
    cols = ['id','parent_id','kind','sha256']
    for ancestor in SOURCE_RUNS:
      for name in variants:
        old_path = ancestor / (name+'_training_ids.csv')
        if not old_path.is_file(): raise FileNotFoundError(f'Нет списка обучения старого запуска: {old_path}')
        old = pd.read_csv(old_path, dtype=str).sort_values('id').reset_index(drop=True)
        new = pd.read_csv(RUN_DIR/(name+'_training_ids.csv'), dtype=str).sort_values('id').reset_index(drop=True)
        fail_if(not old[cols].equals(new[cols]), f'Данные варианта {name} изменились; продолжение остановлено.')
    print('ID и хеши изображений прежнего запуска совпадают. Хеши аннотаций прежний ноутбук не сохранял.')
verify_previous_run()


,вариант,train,validation,test
0,Исходные,2936,367,367
1,Исходные + ручные,3036,367,367
2,Исходные + отражения,5872,367,367
3,Исходные + ручные + отражения,6072,367,367


ID и хеши изображений прежнего запуска совпадают. Хеши аннотаций прежний ноутбук не сохранял.


## 5. Обучение и восстановление

Чекпоинты сохраняются на Google Drive после каждой эпохи. Завершённые модели повторно не обучаются. При обрыве внутри эпохи она будет запущена заново. Для продолжения этого запуска укажите новый ID в настройках и снова выполните все ячейки.


In [ ]:
import torch
import torchvision
import ultralytics
from ultralytics import YOLO
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import pil_to_tensor
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

if not torch.cuda.is_available():
    raise RuntimeError('GPU не обнаружен. Подключите Colab runtime с T4 и выполните ячейки заново.')
DEVICE = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(0))
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED); random.seed(SEED); np.random.seed(SEED)

class DetectionDataset(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(row['path']) as im:
            image = pil_to_tensor(im.convert('RGB')).float() / 255.0
        target = {'boxes':torch.tensor([row['box']], dtype=torch.float32),
                  'labels':torch.tensor([1], dtype=torch.int64),
                  'image_id':torch.tensor(index, dtype=torch.int64)}
        return image, target

def collate_fn(batch): return tuple(zip(*batch))

models_dir = RUN_DIR / 'models'
models_dir.mkdir()
checkpoint_dir = models_dir / 'checkpoints'
checkpoint_dir.mkdir()
model_files = {}
train_log = []
resume_notes = []

# В torch 2.6 загрузка полных чекпоинтов требует weights_only=False.
# Читайте только собственные, доверенные файлы из папки запуска.
def load_checkpoint(path):
    return torch.load(path, map_location='cpu', weights_only=False)

def atomic_torch_save(value, path):
    path = Path(path)
    temp = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    try:
        with temp.open('wb') as f:
            torch.save(value, f)
            f.flush()
            os.fsync(f.fileno())
        os.replace(temp, path)
    finally:
        if temp.exists(): temp.unlink()

def copy_checked(src, dest):
    # Исходную папку не меняем; загрузка обнаруживает повреждённые файлы.
    value = load_checkpoint(src)
    atomic_torch_save(value, dest)
    return value

def verify_yolo(ckpt, name):
    if not isinstance(ckpt, dict) or (ckpt.get('model') is None and ckpt.get('ema') is None):
        raise ValueError(f'Некорректный чекпоинт YOLO {name}.')
    args = ckpt.get('train_args') or {}
    for key, expected_value in [('epochs',YOLO_EPOCHS),('imgsz',IMG_SIZE),('batch',YOLO_BATCH),('seed',SEED)]:
        if args.get(key) != expected_value:
            raise ValueError(f'Настройка YOLO {name}: {key}={args.get(key)}, ожидалось {expected_value}.')

def prepare_yolo_resume_copy(source, name):
    ckpt = load_checkpoint(source)
    verify_yolo(ckpt, name)
    if ckpt.get('optimizer') is None or not isinstance(ckpt.get('epoch'), int) or ckpt['epoch'] < 0:
        raise ValueError(f'YOLO {name}: нет состояния оптимизатора для продолжения: {source}')
    args = dict(ckpt['train_args'])
    args.update(data=str(WORK/(name+'.yaml')), project=str(RUN_DIR/'yolo_runs'), name=name,
                save_dir=str(RUN_DIR/'yolo_runs'/name), exist_ok=True)
    ckpt['train_args'] = args
    dest = RUN_DIR/'resume_inputs'/('yolo_'+name+'.pt')
    dest.parent.mkdir(parents=True, exist_ok=True)
    atomic_torch_save(ckpt, dest)
    history = source.parent.parent/'results.csv'
    if history.is_file():
        target = RUN_DIR/'yolo_runs'/name/'results.csv'
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(history, target)
    return dest, ckpt['epoch'] + 1

def select_faster_checkpoint(name):
    candidates = []
    for source_run in SOURCE_RUNS:
        source_models = source_run/'models'
        candidates.append(source_models/('fasterrcnn_'+name+'.pt'))
        candidates += [source_models/'checkpoints'/f'fasterrcnn_{name}_slot{i}.pt' for i in (0,1)]
    existing = [p for p in candidates if p.is_file()]
    valid, errors = [], []
    for path in existing:
        try:
            ckpt = load_checkpoint(path)
            if not isinstance(ckpt,dict) or ckpt.get('variant') != name or ckpt.get('seed') != SEED:
                raise ValueError('вариант или seed не совпадают')
            epoch = ckpt.get('epoch')
            if type(epoch) is not int or not 1 <= epoch <= FRCNN_EPOCHS:
                raise ValueError('неверный номер эпохи')
            if not isinstance(ckpt.get('model_state_dict'),dict) or not isinstance(ckpt.get('optimizer_state_dict'),dict):
                raise ValueError('нет весов модели или оптимизатора')
            config=ckpt.get('config')
            if config is not None and config != {'img_size':IMG_SIZE,'batch':FRCNN_BATCH,'epochs':FRCNN_EPOCHS}:
                raise ValueError('параметры обучения изменились')
            valid.append((epoch, path))
        except Exception as exc:
            errors.append(f'{path}: {exc}')
    if errors: print('Не удалось прочитать некоторые чекпоинты:', *errors, sep='\n  ', flush=True)
    if existing and not valid:
        raise RuntimeError(f'Все чекпоинты Faster R-CNN {name} повреждены или несовместимы; автоматический перезапуск остановлен.')
    return max(valid, key=lambda x:x[0]) if valid else None

for variant_name, rows in variants.items():
    print('\nYOLO:', variant_labels[variant_name], len(rows), 'изображений', flush=True)
    dest = models_dir / ('yolo_' + variant_name + '.pt')
    completed = next((d/'models'/dest.name for d in SOURCE_RUNS if (d/'models'/dest.name).is_file()), SOURCE_RUN_DIR/'models'/dest.name)
    last = next((d/'yolo_runs'/variant_name/'weights/last.pt' for d in SOURCE_RUNS if (d/'yolo_runs'/variant_name/'weights/last.pt').is_file()), SOURCE_RUN_DIR/'yolo_runs'/variant_name/'weights/last.pt')
    started=time.monotonic()
    if completed.is_file():
        ckpt=load_checkpoint(completed); verify_yolo(ckpt, variant_name)
        atomic_torch_save(ckpt,dest)
        status='reused_completed'
        print('  Готовая модель скопирована:', completed, flush=True)
    else:
        random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
        if last.is_file():
            ckpt=load_checkpoint(last); verify_yolo(ckpt,variant_name)
            if ckpt.get('optimizer') is None:
                # Ultralytics удаляет оптимизатор по окончании обучения; проверяем историю.
                history=ckpt.get('train_results') or {}
                epochs=history.get('epoch', [])
                if len(epochs) < YOLO_EPOCHS or int(epochs[-1]) not in (YOLO_EPOCHS-1, YOLO_EPOCHS):
                    raise RuntimeError(f'YOLO {variant_name}: невозможно подтвердить завершение или продолжить {last}.')
                atomic_torch_save(ckpt,dest); status='reused_completed_last'
                print('  Найдены готовые последние веса:',last,flush=True)
            else:
                resume_path, done = prepare_yolo_resume_copy(last,variant_name)
                # Сохраняем исходную точку и в новом запуске: обрыв возможен до первой новой эпохи.
                carry = RUN_DIR/'yolo_runs'/variant_name/'weights'/'last.pt'
                carry.parent.mkdir(parents=True,exist_ok=True)
                copy_checked(resume_path,carry)
                print(f'  Продолжение после эпохи {done}/{YOLO_EPOCHS}',flush=True)
                model=YOLO(str(resume_path))
                model.train(resume=True, data=str(WORK/(variant_name+'.yaml')), device=0,
                            batch=YOLO_BATCH, imgsz=IMG_SIZE)
                trained_last=Path(model.trainer.save_dir)/'weights/last.pt'
                copy_checked(trained_last,dest); del model; torch.cuda.empty_cache()
                status='resumed'
        else:
            old_cwd=os.getcwd(); os.chdir(WORK)
            try: model=YOLO('yolo11n.pt')
            finally: os.chdir(old_cwd)
            print('  Предыдущих весов нет: обучение с начала.',flush=True)
            model.train(data=str(WORK/(variant_name+'.yaml')), epochs=YOLO_EPOCHS, imgsz=IMG_SIZE,
                        batch=YOLO_BATCH, device=0, workers=2, seed=SEED, deterministic=True,
                        optimizer='SGD', patience=0, save=True, plots=False, verbose=True,
                        project=str(RUN_DIR/'yolo_runs'), name=variant_name, exist_ok=True,
                        mosaic=0.0, mixup=0.0, copy_paste=0.0, fliplr=0.0, flipud=0.0,
                        degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
                        hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, erasing=0.0)
            trained_last=Path(model.trainer.save_dir)/'weights/last.pt'
            copy_checked(trained_last,dest); del model; torch.cuda.empty_cache()
            status='trained'
    model_files[('YOLO11n',variant_name)]=dest
    train_log.append({'model':'YOLO11n','variant':variant_name,'train_count':len(rows),
                      'session_seconds':time.monotonic()-started,'status':status})
    pd.DataFrame(train_log).to_csv(RUN_DIR/'training_times.csv',index=False)

for variant_name, rows in variants.items():
    print('\nFaster R-CNN:',variant_labels[variant_name],len(rows),'изображений',flush=True)
    selected=select_faster_checkpoint(variant_name)
    dest=models_dir/('fasterrcnn_'+variant_name+'.pt')
    started=time.monotonic()
    if selected and selected[0] == FRCNN_EPOCHS:
        copy_checked(selected[1],dest)
        status='reused_completed'
        print('  Готовая модель скопирована:',selected[1],flush=True)
    else:
        random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
        model=fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT,
                    min_size=IMG_SIZE,max_size=IMG_SIZE,box_score_thresh=0.001,box_nms_thresh=0.7,box_detections_per_img=100)
        features=model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor=FastRCNNPredictor(features,2)
        model.to(DEVICE)
        generator=torch.Generator().manual_seed(SEED)
        loader=DataLoader(DetectionDataset(rows),batch_size=FRCNN_BATCH,shuffle=True,
                          num_workers=2,collate_fn=collate_fn,generator=generator)
        optimizer=torch.optim.SGD([p for p in model.parameters() if p.requires_grad],lr=0.005,momentum=0.9,weight_decay=0.0005)
        start_epoch=0
        if selected:
            ckpt=load_checkpoint(selected[1]); start_epoch=selected[0]
            # Копия предыдущей эпохи переживёт обрыв до первого нового сохранения.
            atomic_torch_save(ckpt, checkpoint_dir/f'fasterrcnn_{variant_name}_slot{start_epoch%2}.pt')
            model.load_state_dict(ckpt['model_state_dict']); optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            if all(k in ckpt for k in ('generator_state','torch_rng_state','cuda_rng_state','numpy_rng_state','python_rng_state')):
                generator.set_state(ckpt['generator_state']); torch.set_rng_state(ckpt['torch_rng_state'])
                torch.cuda.set_rng_state_all(ckpt['cuda_rng_state']); np.random.set_state(ckpt['numpy_rng_state'])
                random.setstate(ckpt['python_rng_state'])
            else:
                note=f'{variant_name}: старый чекпоинт без состояния RNG; порядок примеров может отличаться.'
                resume_notes.append(note); print('  Внимание:',note,flush=True)
            print(f'  Загружена эпоха {start_epoch}/{FRCNN_EPOCHS}: {selected[1]}',flush=True)
            status='resumed'
        else:
            print('  Чекпоинтов нет: обучение с начала.',flush=True); status='trained'
        for epoch in range(start_epoch,FRCNN_EPOCHS):
            model.train(); total=0.0
            for step,(images,targets) in enumerate(loader,1):
                images=[image.to(DEVICE) for image in images]
                targets=[{k:v.to(DEVICE) for k,v in target.items()} for target in targets]
                losses=model(images,targets); loss=sum(losses.values())
                if not torch.isfinite(loss): raise RuntimeError(f'Неконечная функция потерь: {variant_name}, epoch {epoch+1}')
                optimizer.zero_grad(); loss.backward(); optimizer.step(); total+=float(loss.detach().cpu())
                if step%200==0 or step==len(loader):
                    print(f'  epoch {epoch+1}/{FRCNN_EPOCHS}: {step}/{len(loader)} batches, loss={total/step:.4f}',flush=True)
            ckpt={'model_state_dict':model.state_dict(),'optimizer_state_dict':optimizer.state_dict(),
                  'epoch':epoch+1,'variant':variant_name,'seed':SEED,
                  'generator_state':generator.get_state(),'torch_rng_state':torch.get_rng_state(),
                  'cuda_rng_state':torch.cuda.get_rng_state_all(),'numpy_rng_state':np.random.get_state(),
                  'python_rng_state':random.getstate(),
                  'config':{'img_size':IMG_SIZE,'batch':FRCNN_BATCH,'epochs':FRCNN_EPOCHS}}
            slot=checkpoint_dir/f'fasterrcnn_{variant_name}_slot{(epoch+1)%2}.pt'
            atomic_torch_save(ckpt,slot)
            print(f'  Сохранена эпоха {epoch+1}/{FRCNN_EPOCHS}: {slot}',flush=True)
        copy_checked(slot,dest)
        del model,loader,optimizer; torch.cuda.empty_cache()
    model_files[('FasterRCNN',variant_name)]=dest
    train_log.append({'model':'FasterRCNN','variant':variant_name,'train_count':len(rows),
                      'session_seconds':time.monotonic()-started,'status':status})
    pd.DataFrame(train_log).to_csv(RUN_DIR/'training_times.csv',index=False)
print('Все восемь моделей готовы к оценке.')


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU: Tesla T4

YOLO: Исходные 2936 изображений
  Готовая модель скопирована: /content/drive/MyDrive/data-enrichment-coursework/results/images/training/20261118T074256Z_475488e0/models/yolo_baseline.pt

YOLO: Исходные + ручные 3036 изображений
  Готовая модель скопирована: /content/drive/MyDrive/data-enrichment-coursework/results/images/training/20261118T074256Z_475488e0/models/yolo_manual.pt

YOLO: Исходные + отражения 5872 изображений
  Готовая модель скопирована: /content/drive/MyDrive/data-enrichment-coursework/results/images/training/20261118T074256Z_475488e0/models/yolo_flipped.pt

YOLO: Исходные + ручные + отражения 6072 изображений
  Готовая модель скопирована: /content/dri

100%|██████████| 160M/160M [00:00<00:00, 189MB/s] 


  Чекпоинтов нет: обучение с начала.
  epoch 1/10: 200/2936 batches, loss=0.1435
  epoch 1/10: 400/2936 batches, loss=0.1189
  epoch 1/10: 600/2936 batches, loss=0.1079
  epoch 1/10: 800/2936 batches, loss=0.1000
  epoch 1/10: 1000/2936 batches, loss=0.0945
  epoch 1/10: 1200/2936 batches, loss=0.0898
  epoch 1/10: 1400/2936 batches, loss=0.0859
  epoch 1/10: 1600/2936 batches, loss=0.0835
  epoch 1/10: 1800/2936 batches, loss=0.0808
  epoch 1/10: 2000/2936 batches, loss=0.0786
  epoch 1/10: 2200/2936 batches, loss=0.0764
  epoch 1/10: 2400/2936 batches, loss=0.0752
  epoch 1/10: 2600/2936 batches, loss=0.0741
  epoch 1/10: 2800/2936 batches, loss=0.0727
  epoch 1/10: 2936/2936 batches, loss=0.0718
  Сохранена эпоха 1/10: /content/drive/MyDrive/data-enrichment-coursework/results/images/training/20261118T103517Z_8430340a/models/checkpoints/fasterrcnn_flipped_slot1.pt
  epoch 2/10: 200/2936 batches, loss=0.0477
  epoch 2/10: 400/2936 batches, loss=0.0450
  epoch 2/10: 600/2936 batches, l

## 6. Общая оценка на test

Обе архитектуры оцениваются одной реализацией COCO: **mAP@0.5** и **mAP@0.5:0.95** по кривой precision–recall. Дополнительно precision, recall и F1 рассчитываются при `confidence ≥ 0.25` и IoU ≥ 0.5; каждый объект сопоставляется не более чем с одной рамкой. В задаче детекции обычная accuracy по изображениям не отражает качество рамок, поэтому она не приводится. Тестовая часть содержит только оригиналы Oxford Pet.

Дополнительно выводится `mean_iou_top1`: для каждого изображения берётся рамка с наибольшей уверенностью не ниже 0,25, считается её IoU с разметкой; если рамки нет, IoU равен 0. `mean_iou_matched` усредняет только правильно сопоставленные рамки с IoU не ниже 0,5.


In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import contextlib
import copy

def coco_ground_truth(rows):
    coco = COCO()
    coco.dataset = {'images':[], 'annotations':[], 'categories':[{'id':1,'name':'pet_head'}], 'info':{}}
    for i,r in enumerate(rows, 1):
        x1,y1,x2,y2=r['box']; coco.dataset['images'].append({'id':i,'width':r['width'],'height':r['height'],'file_name':r['id']})
        coco.dataset['annotations'].append({'id':i,'image_id':i,'category_id':1,'bbox':[x1,y1,x2-x1,y2-y1],
                                            'area':(x2-x1)*(y2-y1),'iscrowd':0})
    with contextlib.redirect_stdout(io.StringIO()): coco.createIndex()
    return coco

def box_iou(a,b):
    x1,y1=max(a[0],b[0]),max(a[1],b[1]);x2,y2=min(a[2],b[2]),min(a[3],b[3])
    inter=max(0,x2-x1)*max(0,y2-y1)
    union=(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter
    return inter/union if union>0 else 0.0

def evaluate_detections(rows, detections, confidence=CONFIDENCE, match_iou=MATCH_IOU):
    # detections: list of {'image_id': int, 'box': [x1,y1,x2,y2], 'score': float}.
    fail_if(len(rows)==0, 'Пустая тестовая выборка.')
    by_image={i:[] for i in range(1,len(rows)+1)}; coco_pred=[]
    for d in detections:
        i=int(d['image_id']); fail_if(i not in by_image, f'Неизвестный image_id: {i}')
        x1,y1,x2,y2=map(float,d['box']); score=float(d['score'])
        fail_if(not all(np.isfinite(v) for v in [x1,y1,x2,y2,score]) or not (0<=score<=1), 'Некорректный прогноз.')
        w,h=rows[i-1]['width'],rows[i-1]['height']
        x1,x2=max(0,min(w,x1)),max(0,min(w,x2)); y1,y2=max(0,min(h,y1)),max(0,min(h,y2))
        if x2<=x1 or y2<=y1: continue
        clean={'image_id':i,'box':[x1,y1,x2,y2],'score':score};by_image[i].append(clean)
        coco_pred.append({'image_id':i,'category_id':1,'bbox':[x1,y1,x2-x1,y2-y1],'score':score})
    tp=fp=fn=0; matched_iou=[]; top1_iou=[]
    for i,row in enumerate(rows,1):
        matched=False
        ordered=sorted(by_image[i], key=lambda x:x['score'], reverse=True)
        eligible=[d for d in ordered if d['score'] >= confidence]
        top1_iou.append(box_iou(eligible[0]['box'], row['box']) if eligible else 0.0)
        for d in ordered:
            if d['score'] < confidence: continue
            iou=box_iou(d['box'],row['box'])
            if not matched and iou>=match_iou:
                tp+=1;matched=True;matched_iou.append(iou)
            else: fp+=1
        if not matched: fn+=1
    precision=tp/(tp+fp) if tp+fp else 0.0; recall=tp/(tp+fn) if tp+fn else 0.0
    f1=2*precision*recall/(precision+recall) if precision+recall else 0.0
    if coco_pred:
        gt=coco_ground_truth(rows)
        with contextlib.redirect_stdout(io.StringIO()):
            dt=gt.loadRes(coco_pred); evaluator=COCOeval(gt,dt,'bbox')
            evaluator.params.imgIds=list(range(1,len(rows)+1));evaluator.evaluate();evaluator.accumulate();evaluator.summarize()
        map50=float(evaluator.stats[1]); map5095=float(evaluator.stats[0])
    else: map50=map5095=0.0
    return {'map50':map50,'map50_95':map5095,'precision':precision,'recall':recall,'f1':f1,
            'tp':tp,'fp':fp,'fn':fn,'mean_iou_top1':float(np.mean(top1_iou)),
            'mean_iou_matched':float(np.mean(matched_iou)) if matched_iou else None}


In [ ]:
# Сначала обучаются все модели; только затем они оцениваются на общем test.
evaluation_rows = []
for (model_name, variant_name), path in model_files.items():
    print('Оценка:', model_name, variant_labels[variant_name], flush=True)
    detections=[]
    if model_name == 'YOLO11n':
        model=YOLO(str(path))
        for start in range(0,len(test_records),16):
            part=test_records[start:start+16]
            results=model.predict(source=[str(r['path']) for r in part], imgsz=IMG_SIZE, device=0,
                                  conf=0.001, iou=0.7, max_det=100, batch=16, verbose=False)
            for j,result in enumerate(results):
                boxes=result.boxes.xyxy.cpu().numpy();scores=result.boxes.conf.cpu().numpy()
                detections.extend({'image_id':start+j+1,'box':box.tolist(),'score':float(score)} for box,score in zip(boxes,scores))
    else:
        model=fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None, min_size=IMG_SIZE, max_size=IMG_SIZE,
                                      box_score_thresh=0.001, box_nms_thresh=0.7, box_detections_per_img=100)
        features=model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor=FastRCNNPredictor(features,2)
        checkpoint=torch.load(path, map_location='cpu', weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict']);model.to(DEVICE);model.eval()
        loader=DataLoader(DetectionDataset(test_records),batch_size=FRCNN_BATCH,shuffle=False,num_workers=2,collate_fn=collate_fn)
        offset=0
        with torch.no_grad():
            for images,_ in loader:
                outputs=model([im.to(DEVICE) for im in images])
                for j,out in enumerate(outputs):
                    boxes=out['boxes'].cpu().numpy(); scores=out['scores'].cpu().numpy();labels=out['labels'].cpu().numpy()
                    detections.extend({'image_id':offset+j+1,'box':box.tolist(),'score':float(score)} for box,score,label in zip(boxes,scores,labels) if int(label)==1)
                offset+=len(images)
    result=evaluate_detections(test_records,detections)
    evaluation_rows.append({'model':model_name,'variant':variant_name,'n_train':len(variants[variant_name]),'n_test':len(test_records),**result})
    (RUN_DIR/f'predictions_{model_name}_{variant_name}.json').write_text(json.dumps(detections,ensure_ascii=False)+'\n',encoding='utf-8')
    del model;torch.cuda.empty_cache()
results_table=pd.DataFrame(evaluation_rows)
base_map=results_table.loc[results_table['variant'].eq('baseline')].set_index('model')['map50_95']
results_table['delta_map50_95_vs_baseline']=results_table['map50_95']-results_table['model'].map(base_map)
results_table['variant_name']=results_table['variant'].map(variant_labels)
results_table.to_csv(RUN_DIR/'test_metrics.csv',index=False)
print('Итоговое сравнение на общем test:')
display(results_table[['model','variant_name','n_train','map50','map50_95','delta_map50_95_vs_baseline',
                       'precision','recall','f1','tp','fp','fn','mean_iou_top1','mean_iou_matched']].round(4))


In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(11,5))
plot_data=results_table.pivot(index='variant_name',columns='model',values='map50_95').reindex([variant_labels[k] for k in variants])
plot_data.plot(kind='bar',ax=ax,color=['#d3854c','#397a78'])
ax.set(title='mAP@0.5:0.95 на общем test',xlabel='',ylabel='mAP',ylim=(0,1));ax.legend(title='Модель')
ax.tick_params(axis='x',rotation=15);fig.tight_layout()
fig.savefig(RUN_DIR/'test_map_comparison.png',dpi=160,bbox_inches='tight');plt.show()


## 7. Сохранение и ограничения

Сохранены общие метрики и отдельные предсказания, веса, время обучения, ID каждого варианта и отдельный файл разбиения. Разные объёмы обучения могут влиять на результат наряду со способом обогащения. Отражение — геометрическая аугментация, не генерация ИИ. Ручные рамки выбраны автоматически по согласованному правилу и не прошли независимую окончательную проверку. Точные хеши не исключают визуальные или смысловые дубликаты. Test не используется для выбора эпох и параметров; после просмотра test не подбирайте настройки по нему.


In [ ]:
import platform
import sklearn
metadata={'run_id':RUN_ID,'created_utc':datetime.now(timezone.utc).isoformat(),'seed':SEED,
          'input_sha256':{key:sha256_file(value) for key,value in PATHS.items() if value.is_file()},
          'split':{'train':len(train_ids),'validation':len(val_ids),'test':len(test_ids),
                   'method':'split original validation 50/50, stratified by breed, seed 42'},
          'variant_counts':{name:len(rows) for name,rows in variants.items()},
          'config':{'img_size':IMG_SIZE,'yolo_epochs':YOLO_EPOCHS,'frcnn_epochs':FRCNN_EPOCHS,
                    'yolo_batch':YOLO_BATCH,'frcnn_batch':FRCNN_BATCH,'confidence':CONFIDENCE,'match_iou':MATCH_IOU,
                    'yolo_weights':'yolo11n.pt','frcnn_weights':'COCO_V1','random_augmentation':False,
                    'checkpoint_selection':'last epoch'},
          'versions':{'python':platform.python_version(),'torch':torch.__version__,'torchvision':torchvision.__version__,
                      'ultralytics':ultralytics.__version__,'numpy':np.__version__,'scikit_learn':sklearn.__version__},
          'limits':['COCO pretrained weights may contain related imagery', 'manual labels are automatically selected',
                    'different augmentation variants use different numbers of training examples',
                    'exact hashes do not detect near duplicates', 'horizontal flips are augmentation, not generative AI']}
metadata['resume'] = {'source_run_id':RESUME_SOURCE_RUN_ID, 'notes':resume_notes,
                      'training_times_are_session_only':True,
                      'legacy_run_has_no_annotation_hashes':True}
(RUN_DIR/'run_metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print('Результаты:',RUN_DIR)
print('test_metrics.csv, prediction JSON, training_times.csv, evaluation_split.csv, training IDs, models/, график и run_metadata.json')
display(results_table[['model','variant_name','n_train','map50','map50_95','precision','recall','f1','mean_iou_top1']].round(4))


## 8. Примеры детекции

Для каждой архитектуры показаны одни и те же три заранее выбранных изображения test. Использован фиксированный вариант «исходные + ручные + отражения», а не выбор лучшего результата по test. Зелёная рамка — разметка, оранжевая — наиболее уверенное предсказание с confidence не ниже 0,25. Если предсказания нет, отмечается «не обнаружено». Для показанной рамки указан её IoU с разметкой.


In [ ]:
from matplotlib.patches import Rectangle, Patch

EXAMPLE_VARIANT = 'manual_flipped'
# Одни и те же ID для обеих моделей, фиксируем seed до просмотра предсказаний.
example_indices = np.random.default_rng(SEED).choice(len(test_records), size=3, replace=False)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for model_index, model_name in enumerate(['YOLO11n', 'FasterRCNN']):
    prediction_path = RUN_DIR / f'predictions_{model_name}_{EXAMPLE_VARIANT}.json'
    example_predictions = json.loads(prediction_path.read_text(encoding='utf-8'))
    for column, index in enumerate(example_indices):
        index = int(index)
        row = test_records[index]
        ax = axes[model_index, column]
        with Image.open(row['path']) as image:
            ax.imshow(image.convert('RGB'))
        x1, y1, x2, y2 = row['box']
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='#27ae60', linewidth=2))
        candidates = sorted((d for d in example_predictions
                             if int(d['image_id']) == index + 1 and float(d['score']) >= CONFIDENCE),
                            key=lambda d: float(d['score']), reverse=True)
        if candidates:
            prediction = candidates[0]
            px1, py1, px2, py2 = map(float, prediction['box'])
            px1, px2 = max(0, min(row['width'], px1)), max(0, min(row['width'], px2))
            py1, py2 = max(0, min(row['height'], py1)), max(0, min(row['height'], py2))
            if px2 > px1 and py2 > py1:
                ax.add_patch(Rectangle((px1, py1), px2-px1, py2-py1, fill=False, edgecolor='#e67e22', linewidth=2))
                iou = box_iou([px1, py1, px2, py2], row['box'])
                detail = f"confidence={float(prediction['score']):.2f}, IoU={iou:.2f}"
            else:
                detail = 'Некорректная рамка'
        else:
            detail = 'Не обнаружено'
        ax.set_title(f"{model_name} · {row['id']}\n{detail}", fontsize=9)
        ax.axis('off')
fig.legend(handles=[Patch(facecolor='none', edgecolor='#27ae60', label='Разметка'),
                    Patch(facecolor='none', edgecolor='#e67e22', label='Предсказание')],
           loc='upper center', ncol=2)
fig.suptitle('Примеры детекции: исходные + ручные + отражения', y=0.99)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.savefig(RUN_DIR / 'detection_examples.png', dpi=160, bbox_inches='tight')
plt.show()
print('Примеры сохранены:', RUN_DIR / 'detection_examples.png')
